# Mutual Information

When presented with a new data set, potentially thousands of features, it is important to have a way to determine which are the most important. Mutual information is a "feature utililty metric" which is a funciton that measures the association between a feature and the target. Mutual information is a good general purpose metric since it can detect any kind of relationship, not just linear.

* easy to use and interpret,
* computationally efficient,
* theoretically well-founded,
* resistant to overfitting, and,
* able to detect any kind of relationship

A great first step is to construct a ranking of the features with the most association, then narrow down the most important and start with those feature to be sure your time is not wasted. Then other features can be added as necessary or if they can be engineered to provide more information.

### What it Measures

Mutual Information (MI) measures uncertainty. How much does knowledge of one feature reduce uncertainty about the target.


Technical notes: 
* What we're calling uncertainty is measured using a quantity from information theory known as "entropy". The entropy of a variable means roughly: "how many yes-or-no questions you would need to describe an occurance of that variable, on average." 
* The more questions you have to ask, the more uncertain you must be about the variable. Mutual information is how many questions you expect the feature to answer about the target.
* Lowest possible mutual information between two quantities is 0.0.
* 0.0 means the variables are independant.
* No strict upper limit but values about 2.0 are uncommon.




![mutual_information](mutual_information_graph.png)

Left: Mutual information increases as the dependence between feature and target becomes tighter. Right: Mutual information can capture any kind of association (not just linear, like correlation.)

Just because a feature has a high MI score, doesn't necessarily mean it will automatically help your model learn or be an association that you model can learn. May still need to expose the relationship to the model through feature engineering.

### Usage

scikit- learn treats discrete features differently from continuous features. We have to specify which is which. Categorical variables need to be converted to integers.

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-whitegrid")

df = pd.read_csv("..data/autos.csv")
df.head()

In [ ]:
X = df.copy()
y = X.pop("price")

# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()

# All discrete features should now have integer dtypes (double-check this before using MI!)
discrete_features = X.dtypes == int

Scikit-learn has two mutual information metrics in its feature_selection module: one for real-valued targets (mutual_info_regression) and one for categorical targets (mutual_info_classif).

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Create a funciton that returns a pandas series of the features and their MI scores.
# This is for discrete features so it uses mutual_info_regression func.
def make_mi_scores(X, y, discrete_features):
    mi_scores = mutual_info_regression(X, y, discrete_features=discrete_features)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores

mi_scores = make_mi_scores(X, y, discrete_features)
mi_scores[::3]  # show a few features with their MI scores

In [ ]:
# Create a plot to see the ranked scores visually
def plot_mi_scores(scores):
    scores = scores.sort_values(ascending=True)
    width = np.arange(len(scores))
    ticks = list(scores.index)
    plt.barh(width, scores)
    plt.yticks(width, ticks)
    plt.title("Mutual Information Scores")


plt.figure(dpi=100, figsize=(8, 5))
plot_mi_scores(mi_scores)

In [ ]:
# Plot the variables features to the target to examine the relationship further.
sns.relplot(x="curb_weight", y="price", data=df);

# Some features with a low MI score can still offer important information, 
# it is good to examine as many feature relationships as possible. Use domain expertise as a guide.
sns.lmplot(x="horsepower", y="price", hue="fuel_type", data=df);

 Combining these top features with other related features, especially those you've identified as creating interactions, is a good strategy for coming up with a highly informative set of features to train your model on.